In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv("D:\GitHub Project\A_DATA ANALYST SIMULATION\DAY 6\IDJuicer_demand_data.csv")

In [3]:
df.head()

,Date,Day_of_Week,is_weekend,pay_day,product_category,product_name,unit_cost,discount_rate,Selling_Price,Total_Units_Sold,Total_Revenue,Profit
0,2025-01-01,Wednesday,0,1,Freshy Coconut,Coconut Water,8000,0.0,18000.0,77,1386000.0,770000.0
1,2025-01-01,Wednesday,0,1,Freshy Coconut,Kelapa Ijo,9000,0.0,20000.0,56,1120000.0,616000.0
2,2025-01-01,Wednesday,0,1,Nut Milk,Choco Blast,19000,0.1,40500.0,44,1782000.0,946000.0
3,2025-01-01,Wednesday,0,1,Nut Milk,Vanilla Shake,19000,0.2,36000.0,29,1044000.0,493000.0
4,2025-01-01,Wednesday,0,1,Juicy Juice,Tropical Vibes,17000,0.0,40000.0,32,1280000.0,736000.0


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2338 entries, 0 to 2337
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Date              2338 non-null   object 
 1   Day_of_Week       2338 non-null   object 
 2   is_weekend        2338 non-null   int64  
 3   pay_day           2338 non-null   int64  
 4   product_category  2338 non-null   object 
 5   product_name      2338 non-null   object 
 6   unit_cost         2338 non-null   int64  
 7   discount_rate     2338 non-null   float64
 8   Selling_Price     2338 non-null   float64
 9   Total_Units_Sold  2338 non-null   int64  
 10  Total_Revenue     2338 non-null   float64
 11  Profit            2338 non-null   float64
dtypes: float64(4), int64(4), object(4)
memory usage: 219.3+ KB


In [5]:
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
from scipy.stats import randint

In [6]:
df_predict = df.copy()

In [7]:
df_predict = df_predict.drop(columns=['Total_Revenue','Profit'])

In [8]:
df_predict['Date'] = pd.to_datetime(df_predict['Date'])
df_predict['Month'] = df_predict['Date'].dt.month
df_predict['Day_of_Year'] = df_predict['Date'].dt.dayofyear
df_predict = df_predict.drop(columns=['Date'])

In [9]:
df_predict.head()

,Day_of_Week,is_weekend,pay_day,product_category,product_name,unit_cost,discount_rate,Selling_Price,Total_Units_Sold,Month,Day_of_Year
0,Wednesday,0,1,Freshy Coconut,Coconut Water,8000,0.0,18000.0,77,1,1
1,Wednesday,0,1,Freshy Coconut,Kelapa Ijo,9000,0.0,20000.0,56,1,1
2,Wednesday,0,1,Nut Milk,Choco Blast,19000,0.1,40500.0,44,1,1
3,Wednesday,0,1,Nut Milk,Vanilla Shake,19000,0.2,36000.0,29,1,1
4,Wednesday,0,1,Juicy Juice,Tropical Vibes,17000,0.0,40000.0,32,1,1


In [10]:
X = df_predict.drop(columns=['Total_Units_Sold'])
y = df_predict['Total_Units_Sold']

In [11]:
# One-hot encode categorical features
X = pd.get_dummies(X, columns=['Day_of_Week', 'product_category', 'product_name'], drop_first=True)

In [12]:
# Split the data into training and testing sets 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [13]:
# Define the parameter distribution
param_dist = {
    'n_estimators': randint(100, 1000), 
    'max_depth': [10, 20, 30, None], 
    'min_samples_split': randint(2, 11),
    'min_samples_leaf': randint(1, 11), 
    'bootstrap': [True, False], 
    'max_features': [1.0, 'sqrt', 'log2'] 
}

In [14]:
# Initialize Random Forest model
rf = RandomForestRegressor(random_state=42, n_jobs=-1)

In [15]:
# Initialize Randomized Search CV
random_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    n_iter=20,
    cv=5,
    scoring='neg_mean_absolute_error', # Optimize for minimizing MAE
    verbose=2,
    random_state=42,
    n_jobs=-1
)

In [16]:
# Model Training (Tuning) 
print("Starting Randomized Search...")
random_search.fit(X_train, y_train)

# Get the best estimator
best_rf = random_search.best_estimator_

Starting Randomized Search...
Fitting 5 folds for each of 20 candidates, totalling 100 fits


In [17]:
# Evaluation of the Best Model 
y_pred = best_rf.predict(X_test)

# R2 Score
r2 = r2_score(y_test, y_pred)

# MAE (Mean Absolute Error)
mae = mean_absolute_error(y_test, y_pred)

# RMSE (Root Mean Square Error)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

# MAPE (Mean Absolute Percentage Error)
# Custom calculation: (1/n) * sum(|(y_i - y_hat_i) / y_i|) * 100
# use a small epsilon to avoid division by zero, although Total_Units_Sold should be > 0.
epsilon = 1e-10
mape = np.mean(np.abs((y_test - y_pred) / (y_test + epsilon))) * 100


In [18]:
# Output Results 
print("\n--- Regression Prediction Scores (Tuned Model) ---")
print(f"R-squared (R2): {r2:.4f}")
print(f"Mean Absolute Error (MAE): {mae:.4f} units")
print(f"Root Mean Square Error (RMSE): {rmse:.4f} units")
print(f"Mean Absolute Percentage Error (MAPE): {mape:.2f}%")


--- Regression Prediction Scores (Tuned Model) ---
R-squared (R2): 0.9513
Mean Absolute Error (MAE): 4.2420 units
Root Mean Square Error (RMSE): 7.1846 units
Mean Absolute Percentage Error (MAPE): 9.65%


In [19]:
scores_data = {
    'Metric': ['R-squared (R2)', 'Mean Absolute Error (MAE)', 'Root Mean Square Error (RMSE)', 'Mean Absolute Percentage Error (MAPE)'],
    'Value': [r2, mae, rmse, mape],
    'Unit': ['(none)', 'units', 'units', '%']
}

scores_df = pd.DataFrame(scores_data)
scores_df.head()

,Metric,Value,Unit
0,R-squared (R2),0.951318,(none)
1,Mean Absolute Error (MAE),4.242040,units
2,Root Mean Square Error (RMSE),7.184553,units
3,Mean Absolute Percentage Error (MAPE),9.649218,%


In [20]:
best_params = {'bootstrap': True, 'max_depth': 20, 'max_features': 1.0, 'min_samples_leaf': 2, 'min_samples_split': 6, 'n_estimators': 891}

best_rf = RandomForestRegressor(random_state=42, n_jobs=-1, **best_params)
best_rf.fit(X_train, y_train)

RandomForestRegressor(max_depth=20, min_samples_leaf=2, min_samples_split=6,
                      n_estimators=891, n_jobs=-1, random_state=42)

In [21]:
import joblib
# Save the trained model object
joblib.dump(best_rf, 'demand_model.joblib')

# Save the list of feature columns (CRUCIAL for consistent feature ordering in deployment)
feature_columns = X.columns.tolist()
joblib.dump(feature_columns, 'model_features.joblib')

print("Final model saved to 'demand_model.joblib'.")
print("Feature column list saved to 'model_features.joblib'.")

Final model saved to 'demand_model.joblib'.
Feature column list saved to 'model_features.joblib'.
